# Fine-tune Face Recognition (AdaFace IR-50) tren Google Colab

Notebook nay fine-tune model nhan dien khuon mat cho he thong cham cong.

**Quy trinh:**
1. Kiem tra GPU
2. Mount Google Drive (luu checkpoint, tranh mat khi disconnect)
3. Upload code recognition (`recognition_colab.zip`)
4. Cai dependencies
5. Tai pretrained AdaFace
6. Tai dataset CASIA-WebFace
7. Chan doan (verify pretrained load dung)
8. Fine-tune (~1-2h tren T4)
9. Danh gia LFW
10. Tai checkpoint ve / luu Drive

**LUU Y QUAN TRONG:**
- Bat GPU: Runtime -> Change runtime type -> Hardware accelerator -> **GPU (T4)**
- Colab free co the disconnect sau ~90 phut idle. Checkpoint duoc luu vao Drive moi epoch nen co the resume.


In [ ]:
!pip install -q pytorch_lightning


## Buoc 1 — Kiem tra GPU

Phai thay GPU (Tesla T4). Neu khong co, vao Runtime -> Change runtime type -> GPU.

In [ ]:
!nvidia-smi
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Fri Jun  5 14:05:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Buoc 2 — Mount Google Drive

Checkpoint se duoc luu vao Drive de:
- Khong mat khi Colab disconnect
- Co the resume training
- Tai ve may sau khi xong

Chay cell duoi, bam link, dang nhap Google, dan code xac thuc.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Thu muc luu checkpoint tren Drive
CKPT_DIR = '/content/drive/MyDrive/attendance/checkpoints/recognition'
os.makedirs(CKPT_DIR, exist_ok=True)
print("Checkpoint se luu tai:", CKPT_DIR)

Mounted at /content/drive
Checkpoint se luu tai: /content/drive/MyDrive/attendance/checkpoints/recognition


## Buoc 3 — Upload code recognition

Chay cell duoi -> bam "Choose Files" -> chon file **`recognition_colab.zip`** (file minh gui).

Sau khi upload xong, code se duoc giai nen vao `/content/recognition/`.

In [ ]:
from google.colab import files
import zipfile, os

print("Chon file recognition_colab.zip de upload...")
uploaded = files.upload()  # chon recognition_colab.zip

os.makedirs('/content/recognition', exist_ok=True)
for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as z:
            z.extractall('/content/recognition')
        print(f"Da giai nen {fname} vao /content/recognition/")

# Them vao Python path de import duoc
import sys
sys.path.insert(0, '/content/recognition')
print("\nCac file code:")
!ls -la /content/recognition/

Chon file recognition_colab.zip de upload...


Saving recognition_colab.zip to recognition_colab.zip
Da giai nen recognition_colab.zip vao /content/recognition/

Cac file code:
total 60
drwxr-xr-x 2 root root  4096 Jun  5 14:07 .
drwxr-xr-x 1 root root  4096 Jun  5 14:07 ..
-rw-r--r-- 1 root root  2838 Jun  5 14:07 adaface.py
-rw-r--r-- 1 root root  2980 Jun  5 14:07 dataset.py
-rw-r--r-- 1 root root  6067 Jun  5 14:07 diagnose_recognition.py
-rw-r--r-- 1 root root  3319 Jun  5 14:07 download_pretrained.py
-rw-r--r-- 1 root root  4765 Jun  5 14:07 evaluate.py
-rw-r--r-- 1 root root  3752 Jun  5 14:07 extract_embedding.py
-rw-r--r-- 1 root root 10045 Jun  5 14:07 finetune.py
-rw-r--r-- 1 root root  7334 Jun  5 14:07 iresnet.py


## Buoc 4 — Cai dependencies

PyTorch da co san tren Colab. Chi can them vai package nhe.
(KHONG can mediapipe vi do la phan inference chay o may local.)

In [ ]:
!pip install -q gdown scikit-learn opencv-python-headless
print("Done installing dependencies")

Done installing dependencies


## Buoc 5 — Tai pretrained AdaFace IR-50 (MS1MV2)

Tai trong so pretrained (~250MB) tu Google Drive.

Neu link gdown bi loi (rate limit), xem phan **Phuong an khac** ben duoi.

In [ ]:
import os
os.makedirs('/content/pretrained', exist_ok=True)
ckpt_path = '/content/pretrained/adaface_ir50_ms1mv2.ckpt'

if os.path.exists(ckpt_path):
    print("File da ton tai, bo qua download.")
else:
    # AdaFace ir50 ms1mv2 official
    !gdown 1eUaSHG4pGlIZK7hBkqjyp2fc2epKoBvI -O {ckpt_path}

import os
if os.path.exists(ckpt_path):
    sz = os.path.getsize(ckpt_path) / (1024*1024)
    print(f"\n[OK] Pretrained: {sz:.1f} MB")
else:
    print("[FAIL] Download that bai. Xem phuong an khac ben duoi.")

Downloading...
From (original): https://drive.google.com/uc?id=1eUaSHG4pGlIZK7hBkqjyp2fc2epKoBvI
From (redirected): https://drive.google.com/uc?id=1eUaSHG4pGlIZK7hBkqjyp2fc2epKoBvI&confirm=t&uuid=c6209b15-23a3-4a9a-a283-b462098ca31c
To: /content/pretrained/adaface_ir50_ms1mv2.ckpt
100% 700M/700M [00:13<00:00, 50.9MB/s]

[OK] Pretrained: 667.8 MB


**Phuong an khac neu gdown loi:**

Tai tay tu HuggingFace roi upload, hoac dung huggingface_hub:
```python
!pip install -q huggingface_hub
from huggingface_hub import hf_hub_download
p = hf_hub_download(repo_id="VishalMishraTss/AdaFace",
                    filename="adaface_ir50_ms1mv2.ckpt",
                    local_dir="/content/pretrained")
print(p)
```

## Buoc 6 — Tai dataset CASIA-WebFace (da align 112x112)

Dung Kaggle API. Can file **`kaggle.json`**:
1. Vao https://www.kaggle.com -> Settings -> API -> "Create New Token"
2. Tai ve file kaggle.json
3. Chay cell duoi, upload kaggle.json

Dataset ~3.5GB, tai mat vai phut.

In [ ]:
from google.colab import files
import os, json

print("Upload file kaggle.json...")
uploaded = files.upload()  # chon kaggle.json

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
!pip install -q kaggle
print("Kaggle API ready")

Upload file kaggle.json...


Saving kaggle.json to kaggle.json
Kaggle API ready


In [ ]:
# Tai CASIA-WebFace aligned tu Kaggle
# Neu dataset nay khong ton tai, search tren Kaggle: "casia webface aligned 112"
# va thay slug ben duoi (dang: username/dataset-name)
!kaggle datasets download -d ntl0601/casia-webface -p /content --unzip

# Kiem tra cau truc (tim thu muc chua cac folder identity)
import os
for root, dirs, files_ in os.walk('/content'):
    # Tim thu muc co nhieu sub-folder (identities)
    if len(dirs) > 1000:
        print(f"Tim thay dataset tai: {root} ({len(dirs)} identities)")
        break
!ls /content | head

Dataset URL: https://www.kaggle.com/datasets/ntl0601/casia-webface
License(s): unknown
100% 2.53G/2.53G [00:40<00:00, 66.7MB/s]

Tim thay dataset tai: /content/casia-webface (10572 identities)
casia-webface
casia-webface.txt
drive
kaggle.json
pretrained
recognition
recognition_colab.zip
sample_data


**Luu y ve duong dan dataset:**

Sau khi giai nen, cau truc co the la `/content/casia_webface_aligned/` hoac ten khac.
Chay cell duoi de tim dung duong dan, roi dat vao bien `DATA_DIR`.

In [ ]:
import os
DATA_DIR = '/content/casia-webface'

# Verify cau truc
ids = sorted(os.listdir(DATA_DIR))
print(f"So identities: {len(ids)}")
sample = os.path.join(DATA_DIR, ids[0])
print(f"Folder dau tien: {sample}")
print(f"Noi dung mau: {os.listdir(sample)[:5]}")

So identities: 10572
Folder dau tien: /content/casia-webface/000000
Noi dung mau: ['00000012.jpg', '00000010.jpg', '00000005.jpg', '00000002.jpg', '00000011.jpg']


## Buoc 7 — Chan doan (verify pretrained load dung)

QUAN TRONG: chay truoc khi train de chac chan pretrained khop kien truc.
Can vai anh mat. Notebook se dung anh tu chinh dataset CASIA de test.

In [ ]:
import os, random
# Lay 3 anh: 2 cung 1 nguoi, 1 nguoi khac
ids = sorted(os.listdir(DATA_DIR))
id1 = os.path.join(DATA_DIR, ids[0])
id2 = os.path.join(DATA_DIR, ids[1])
imgs1 = sorted(os.listdir(id1))
face_a = os.path.join(id1, imgs1[0])
face_b = os.path.join(id1, imgs1[1]) if len(imgs1) > 1 else face_a
face_c = os.path.join(id2, sorted(os.listdir(id2))[0])
print("face_a, face_b = cung 1 nguoi")
print("face_c = nguoi khac")

!cd /content/recognition && python diagnose_recognition.py --pretrained /content/pretrained/adaface_ir50_ms1mv2.ckpt --faces "{face_a}" "{face_b}" "{face_c}" --device cuda

face_a, face_b = cung 1 nguoi
face_c = nguoi khac
========= CHECK 1: Does PRETRAINED load correctly? =========
Backbone:        ir_50
Total keys:      467
Loaded keys:     467
Missing keys:    0
Unexpected keys: 2

  [OK] Pretrained loaded well (architecture matches).

===== CHECK 2: Do PRETRAINED embeddings discriminate? ======
  Embedding norms (raw, before normalize would be ~20):
    [0] 00000001.jpg         L2=1.0000  first3=[-0.05189171  0.00365948  0.00665824]
    [1] 00000002.jpg         L2=1.0000  first3=[-0.02368697  0.00355784  0.00075019]
    [2] 00000016.jpg         L2=1.0000  first3=[-0.03500694 -0.03483482  0.08344103]

  Pairwise cosine similarity:
                    [0]  [1]  [2]
  [0] 00000001.jpg    1.00  0.72  -0.02
  [1] 00000002.jpg    0.72  1.00  -0.04
  [2] 00000016.jpg    -0.02  -0.04  1.00

========================= VERDICT ==========================
  Expected for GOOD model:
    same person pair   -> 0.4 to 0.8
    different person   -> -0.1 to 0.25
  If AL

**Doc ket qua:**
- CHECK 1: `Missing keys: 0` -> pretrained load dung. Neu >200 -> sai file checkpoint.
- CHECK 2: face_a vs face_b (cung nguoi) sim CAO, face_a vs face_c (khac) sim THAP.

Neu OK -> sang Buoc 8 train. Neu sai -> bao lai.

## Buoc 8 — Fine-tune

Train 5 epochs, LR 1e-4. Checkpoint luu vao Drive moi epoch.
Tren T4: batch 128 duoc, ~20-30 phut/epoch -> ~1.5-2.5h tong.

Neu OOM (out of memory): giam `--batch` xuong 64.

In [ ]:
# Fine-tune. Output luu thang vao Drive de khong mat.
!cd /content/recognition && python finetune.py \
    --data "{DATA_DIR}" \
    --pretrained /content/pretrained/adaface_ir50_ms1mv2.ckpt \
    --backbone ir_50 \
    --batch 128 \
    --epochs 5 \
    --lr 1e-4 \
    --workers 2 \
    --amp \
    --out "{CKPT_DIR}"

print("\n=== Train xong. Checkpoint tai:", CKPT_DIR)
!ls -la "{CKPT_DIR}"

Device: cuda
GPU: Tesla T4 (14.6 GB)
Dataset: 490623 images / 10572 identities
Loading pretrained: /content/pretrained/adaface_ir50_ms1mv2.ckpt
  missing keys:    0
  unexpected keys: 2
Trainable params: 49.00M
/content/recognition/finetune.py:169: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() if (args.amp and device.type == "cuda") else None

=== Epoch 1/5 ===
/content/recognition/finetune.py:206: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):
  E1 step 50/3832  loss=38.8013  acc=0.0000  lr=0.000001  norm_mean=22.51  speed=238 img/s
  E1 step 100/3832  loss=38.8104  acc=0.0000  lr=0.000003  norm_mean=22.52  speed=239 img/s
  E1 step 150/3832  loss=38.8058  acc=0.0000  lr=0.000004  norm_mean=22.49  speed=232 img/s
  E1 step 200/3832  

**Dau hieu train DUNG (lan nay):**
- Dau ra `missing keys: 0` (pretrained load OK)
- Loss bat dau THAP (~5-15), KHONG phai 38
- Acc tang nhanh ngay epoch 1

**Neu Colab disconnect giua chung:**
Chay lai cell tren, them `--resume "{CKPT_DIR}/last.pt"` de tiep tuc tu checkpoint da luu.

## Buoc 9 — Danh gia LFW (tuy chon)

Can dataset LFW (anh aligned + pairs.txt). Neu chua co, co the bo qua va danh gia sau o may local.

Tai LFW tu Kaggle hoac bo qua buoc nay.

In [ ]:
# Neu co LFW dataset, chay eval. Vi du (sua duong dan cho dung):
# !cd /content/recognition && python evaluate.py \
#     --ckpt "{CKPT_DIR}/last.pt" \
#     --root /content/lfw_aligned_112 \
#     --pairs /content/lfw_pairs.txt \
#     --device cuda
print("Bo qua eval o Colab cung duoc — co the eval o may local sau.")

Bo qua eval o Colab cung duoc — co the eval o may local sau.


## Buoc 10 — Tai checkpoint ve may

Checkpoint da o trong Google Drive (`MyDrive/attendance/checkpoints/recognition/last.pt`).

**Cach 1:** Tai truc tiep tu cell nay.
**Cach 2:** Mo Google Drive tren may, tai `last.pt` ve.

In [ ]:
from google.colab import files
import os

ckpt_file = os.path.join(CKPT_DIR, 'last.pt')
if os.path.exists(ckpt_file):
    sz = os.path.getsize(ckpt_file) / (1024*1024)
    print(f"Tai last.pt ({sz:.1f} MB)...")
    files.download(ckpt_file)
else:
    print("Khong thay last.pt. Kiem tra lai buoc train.")

Tai last.pt (374.2 MB)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Xong!

Sau khi tai `last.pt` ve may:
1. Dat vao `D:\attendance-camera\checkpoints\recognition\last.pt`
2. Test bang `recognize_test.py` o may local
3. Tiep tuc Phase 3 (Anti-Spoofing)

**Luu y:** File `last.pt` train tren Colab dung duoc o may local — model khong phu thuoc may.